# Map Visualization

## Goal

Visualize resolved locations on an interactive Folium map.

## What you will do

- Load resolved outputs if available.
- Validate coordinates.
- Create an interactive map.
- Save it to `outputs/maps/sample_geoparsing_map.html`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import RESULTS_DIR, MAPS_DIR
from src.data_utils import load_dataframe_if_exists
from src.visualization_utils import validate_coordinates, create_location_map, save_map

## Step 1: Load available result files

In [ ]:
candidate_files = [
    RESULTS_DIR / 'unitoprank_results.csv',
    RESULTS_DIR / 'llm_rag_results.csv',
    RESULTS_DIR / 'geocoder_candidates.csv',
]
df = None
for path in candidate_files:
    df = load_dataframe_if_exists(path)
    if df is not None and not df.empty:
        print('Loaded:', path)
        break
if df is None or df.empty:
    df = pd.DataFrame([
        {'mention': 'Paris', 'selected_name': 'Paris', 'country': 'France', 'lat': 48.8566, 'lon': 2.3522, 'method': 'built_in_example'},
        {'mention': 'Berlin', 'selected_name': 'Berlin', 'country': 'Germany', 'lat': 52.52, 'lon': 13.405, 'method': 'built_in_example'},
    ])
df.head()

## Step 2: Validate coordinates

In [ ]:
valid = validate_coordinates(df)
valid

## Step 3: Choose popup columns

Popups should show enough context to interpret the point without making the marker hard to read.

In [ ]:
popup_cols = [col for col in ['mention', 'selected_name', 'name', 'country', 'method'] if col in valid.columns]
popup_cols

## Step 4: Create the map

In [ ]:
fmap = create_location_map(valid, popup_cols=popup_cols)
fmap

## Step 5: Save the map

In [ ]:
out = save_map(fmap, MAPS_DIR / 'sample_geoparsing_map.html')
print('Saved:', out)

## Exercise

Change the popup columns or visualize only one resolution method.

In [ ]:
if 'method' in valid.columns:
    one_method = valid[valid['method'] == valid['method'].iloc[0]]
    create_location_map(one_method, popup_cols=popup_cols)

## Common issues

- Rows without valid latitude and longitude are skipped.
- If there are no valid coordinates, the map opens at a world view.
- Folium maps are saved as standalone HTML files.